# 05 — Shor Code, CSS Codes, and the Steane Code

## Read before you begin

- Arthur Pesah, [*Stabilizer Formalism I*](https://arthurpesah.me/blog/2023-01-31-stabilizer-formalism-1/).
- Sections to read: **Our first truly quantum code: Shor's code**, **Stabilizer formalism: first definitions**, **Quantum codes from classical codes: the CSS construction**, **Steane code, the quantum version of the Hamming code**.
- Concepts to extract: how Shor's code nests a bit-flip code inside a phase-flip code; the general definition of a stabilizer group and code space (beyond the two-generator repetition-code case from notebook 03); the CSS recipe for turning classical parity checks into commuting quantum stabilizers; why the Hamming code specifically can generate *both* the $X$-type and $Z$-type stabilizers of a quantum code.

This notebook does not re-derive what a stabilizer or a syndrome is — notebook 03 already covers that. It builds directly on those definitions at a larger scale, using Stim's `TableauSimulator` — a genuine stabilizer-circuit simulator — instead of hand-built NumPy matrices for the eigenvalue bookkeeping. (Nothing here assumes any later notebook; Stim's own `Circuit`-building tools, used the same way as notebooks 01 and 03, are all that's needed.)

## Learning objectives

By the end of this notebook you should be able to:

1. build Shor's $[[9,1,3]]$ code with an actual encoding circuit (Stim gates, run through `stim.TableauSimulator`) and read off its 8 stabilizer generators;
2. detect a bit-flip and a phase-flip error by checking real stabilizer eigenvalues after applying the error to the simulator, and explain *degeneracy* — why several different physical errors leave the *identical* resulting quantum state;
3. verify computationally that the Shor code is $[[9,1,3]]$: 1 logical qubit from 9 physical qubits, distance 3;
4. state the general CSS construction and explain why it needs a classical code whose parity checks are pairwise orthogonal;
5. build the Steane code's encoding circuit directly from the $[7,4,3]$ Hamming code's parity-check matrix (notebook 04), verify its $X$- and $Z$-type stabilizers commute using Stim's own `PauliString.commutes()`, and read off single-error syndromes.

**Expected outputs:** a verified 8-stabilizer Shor-code encoding circuit and a verified 6-stabilizer Steane-code encoding circuit (both real Stim circuits), an explicit degeneracy demonstration comparing real simulator states, a computational proof of $[[9,1,3]]$, and a Steane-code syndrome table matching notebook 04's classical one exactly (since it reuses the identical $H$).

**Assumes:** notebook 03 (stabilizers, syndromes, eigenvalues, Pauli $X$/$Z$) and notebook 04 (the $[7,4,3]$ Hamming code's $H$ matrix, reused here verbatim).

## 1. `stim.TableauSimulator`: a real stabilizer simulator

Notebook 01 built circuits and *sampled* them (`compile_sampler().sample(...)`), useful for running many random shots. This notebook instead builds a circuit *interactively*, one gate at a time, on a `stim.TableauSimulator` — a live stabilizer-state simulator that can answer "what is the current eigenvalue of this Pauli operator?" directly, via `peek_observable_expectation`. This is the same question notebook 03's hand-built `eigenvalue(op, state)` function answered with NumPy matrices; here Stim answers it using its own internal (and far more scalable) stabilizer-tracking representation.

In [1]:
import stim

sim = stim.TableauSimulator()
sim.h(0)  # qubit 0 -> |+>

print("<Z0> on |+>:", sim.peek_observable_expectation(stim.PauliString("Z")))
print("<X0> on |+>:", sim.peek_observable_expectation(stim.PauliString("X")))

<Z0> on |+>: 0
<X0> on |+>: 1


As expected from notebook 03: $|+\rangle$ is an eigenstate of $X$ (eigenvalue $+1$), not of $Z$ (eigenvalue $0$ — Stim reports $0$ when the state is *not* an eigenstate of the given operator at all, rather than $\pm1$).

## 2. Shor's code: nesting bit-flip protection inside phase-flip protection

Shor's code arranges 9 physical qubits into 3 blocks of 3. The **encoding circuit** for $|0_L\rangle$, built directly from gates:

1. `CX(0,3)`, `CX(0,6)` — copy qubit 0's value onto the other two block leaders (qubits 3 and 6). Starting from $|0\rangle$ this does nothing yet, but it is the step that would spread a general logical value across all three blocks.
2. `H` on qubits 0, 3, 6 — put each block leader into an equal superposition. This is the *outer*, phase-flip-protecting layer (compare $|+\rangle$/$|-\rangle$ playing the role $|0\rangle$/$|1\rangle$ played in notebook 02's classical repetition code).
3. `CX(0,1)`, `CX(0,2)`, and the matching pairs for blocks B and C — spread each block leader's value to its two partners. This is the *inner*, bit-flip-protecting layer: exactly notebook 02's 3-bit repetition code, applied independently within each block.

In [2]:
def build_shor_encoder():
    sim = stim.TableauSimulator()
    sim.cx(0, 3)
    sim.cx(0, 6)
    sim.h(0)
    sim.h(3)
    sim.h(6)
    sim.cx(0, 1)
    sim.cx(0, 2)
    sim.cx(3, 4)
    sim.cx(3, 5)
    sim.cx(6, 7)
    sim.cx(6, 8)
    return sim


shor_generator_strings = [
    "ZZIIIIIII",  # block A: Z0 Z1
    "IZZIIIIII",  # block A: Z1 Z2
    "IIIZZIIII",  # block B: Z3 Z4
    "IIIIZZIII",  # block B: Z4 Z5
    "IIIIIIZZI",  # block C: Z6 Z7
    "IIIIIIIZZ",  # block C: Z7 Z8
    "XXXXXXIII",  # blocks A+B: X0..X5
    "IIIXXXXXX",  # blocks B+C: X3..X8
]

shor_sim = build_shor_encoder()
for s in shor_generator_strings:
    print(f"{s}  eigenvalue: {shor_sim.peek_observable_expectation(stim.PauliString(s)):+.0f}")

ZZIIIIIII  eigenvalue: +1
IZZIIIIII  eigenvalue: +1
IIIZZIIII  eigenvalue: +1
IIIIZZIII  eigenvalue: +1
IIIIIIZZI  eigenvalue: +1
IIIIIIIZZ  eigenvalue: +1
XXXXXXIII  eigenvalue: +1
IIIXXXXXX  eigenvalue: +1


All 8 eigenvalues are $+1$: this circuit really does prepare a state stabilized by all 8 generators, exactly as notebook 03's definition of a stabilizer requires.

### Prediction 1

**Before running the next cell:** an $X_4$ error (qubit 4 is the middle qubit of block B) should flip exactly the $Z$-type checks that touch qubit 4 — which two of the six? An $X$ error never flips an $X$-type check (same-letter Paulis always commute) — so what should the two $X$-type stabilizer eigenvalues be after $X_4$? Now do the same reasoning for a $Z_4$ error, but for the $X$-type checks instead.

In [3]:
def eigenvalues_after_error(qubit, kind, generator_strings):
    sim = build_shor_encoder()
    getattr(sim, kind.lower())(qubit)  # sim.x(qubit) or sim.z(qubit)
    return [sim.peek_observable_expectation(stim.PauliString(s)) for s in generator_strings]


print("Generators:      ", shor_generator_strings)
print("Eigenvalues after X4:", eigenvalues_after_error(4, "X", shor_generator_strings))
print("Eigenvalues after Z4:", eigenvalues_after_error(4, "Z", shor_generator_strings))

Generators:       ['ZZIIIIIII', 'IZZIIIIII', 'IIIZZIIII', 'IIIIZZIII', 'IIIIIIZZI', 'IIIIIIIZZ', 'XXXXXXIII', 'IIIXXXXXX']
Eigenvalues after X4: [1, 1, -1, -1, 1, 1, 1, 1]
Eigenvalues after Z4: [1, 1, 1, 1, 1, 1, -1, -1]


As predicted: $X_4$ flips only the two $Z$-type checks that include qubit 4 (block B's internal checks) — the $X$-type checks are untouched because $X$ always commutes with $X$. $Z_4$ does the opposite: it flips both $X$-type checks (both include qubit 4, since block B sits inside both 6-qubit spans) and leaves every $Z$-type check alone. This reproduces notebook 02/03's 2-bit-syndrome pattern, but now applied to *blocks* rather than individual bits.

## 3. Degeneracy: why a block-level correction is enough

A $Z$ error can land on *any* of the 3 qubits in a block. Does it matter which one? Rather than compare eigenvalues (which only tells us the *syndrome* matches), we compare the actual resulting quantum states directly, via `state_vector()` — the strongest possible notion of "the same error."

In [4]:
import numpy as np

base_sim = build_shor_encoder()

sim_z0 = base_sim.copy(); sim_z0.z(0)
sim_z1 = base_sim.copy(); sim_z1.z(1)
sim_z2 = base_sim.copy(); sim_z2.z(2)

v0, v1, v2 = sim_z0.state_vector(), sim_z1.state_vector(), sim_z2.state_vector()

# Compare up to an overall (unobservable) global phase.
def same_state(a, b):
    return np.isclose(abs(np.vdot(a, b)), 1.0)

print("Z0 and Z1 produce the identical state:", same_state(v0, v1))
print("Z0 and Z2 produce the identical state:", same_state(v0, v2))

Z0 and Z1 produce the identical state: True
Z0 and Z2 produce the identical state: True


$Z_0$, $Z_1$, and $Z_2$ don't just give the same *syndrome* — `sim.copy()` let us branch the simulation and confirm they produce the exact same resulting quantum state. This is **degeneracy**: multiple distinct physical errors are, as far as the code is concerned, the same error. A decoder never needs to figure out *which* qubit in block A had the phase flip — it only needs to know *that* block A did, and can apply a fixed, arbitrary representative correction (say, always $Z_0$) to fully repair the state.

## 4. Proving Shor's code is $[[9,1,3]]$

The notation $[[n,k,d]]$ means: $n=9$ physical qubits, $k$ logical qubits, distance $d$.

**$d=3$:** two weight-3 operators already do the job of logical operators. Recall $Z_0Z_3Z_6$ (one $Z$ per block) should flip every block's sign, mapping $|0_L\rangle \leftrightarrow |1_L\rangle$ — a logical $\bar X$. And $X_0X_1X_2$ (all of block A) should distinguish $|0_L\rangle$ from $|1_L\rangle$ by sign — a logical $\bar Z$.

In [5]:
zero_L = build_shor_encoder().state_vector()

x_bar_sim = build_shor_encoder()
x_bar_sim.z(0); x_bar_sim.z(3); x_bar_sim.z(6)
x_bar_applied = x_bar_sim.state_vector()

z_bar_sim = build_shor_encoder()
print("Z_bar (X0X1X2) eigenvalue on |0_L>:", z_bar_sim.peek_observable_expectation(stim.PauliString("XXX" + "I" * 6)))

# X_bar should map |0_L> to a state orthogonal to |0_L> (a different logical state), not equal to it.
print("X_bar |0_L> is a different state from |0_L> (up to phase):", not same_state(zero_L, x_bar_applied))

Z_bar (X0X1X2) eigenvalue on |0_L>: 1
X_bar |0_L> is a different state from |0_L> (up to phase): True


Both are genuine weight-3 logical operators, so $d \le 3$. To show $d \ge 3$ (no weight-1 or weight-2 operator can act as a nontrivial logical operator), we exhaustively test every weight-1 and weight-2 Pauli string, using Stim's own `PauliString.commutes()` in place of a hand-rolled commutation check.

In [6]:
from itertools import combinations, product as iproduct

N_QUBITS = 9
X_bar = stim.PauliString("Z" + "I" * 2 + "Z" + "I" * 2 + "Z" + "I" * 2)  # Z0 Z3 Z6
Z_bar = stim.PauliString("XXX" + "I" * 6)                                # X0 X1 X2
shor_generators = [stim.PauliString(s) for s in shor_generator_strings]


def all_weight_le(max_weight, n=N_QUBITS):
    strings = []
    for weight in range(1, max_weight + 1):
        for positions in combinations(range(n), weight):
            for letters in iproduct("XYZ", repeat=weight):
                chars = ["I"] * n
                for pos, letter in zip(positions, letters):
                    chars[pos] = letter
                strings.append("".join(chars))
    return strings


def is_trivial(pauli_string):
    """A commuting operator is trivial (a stabilizer) iff it also commutes with
    both logical operators -- anticommuting with either means it changes what a
    logical measurement would read, i.e. it IS a nontrivial logical operator."""
    p = stim.PauliString(pauli_string)
    return p.commutes(X_bar) and p.commutes(Z_bar)


candidates = all_weight_le(2)
nontrivial_low_weight = []
for candidate in candidates:
    p = stim.PauliString(candidate)
    if all(p.commutes(g) for g in shor_generators) and not is_trivial(candidate):
        nontrivial_low_weight.append(candidate)

print(f"Checked {len(candidates)} weight-1 and weight-2 Pauli strings.")
print("Nontrivial logical operators found at weight <= 2:", nontrivial_low_weight)
print("Distance is therefore exactly 3:", len(nontrivial_low_weight) == 0)

Checked 351 weight-1 and weight-2 Pauli strings.
Nontrivial logical operators found at weight <= 2: []
Distance is therefore exactly 3: True


No weight-1 or weight-2 operator escapes detection *and* acts nontrivially on the logical qubit, while Section 4 already exhibited weight-3 operators that do. **The Shor code is exactly $[[9,1,3]]$.**

## 5. The CSS construction: quantum codes from classical codes

The **CSS construction** (Calderbank–Shor–Steane) is a general recipe: take a classical linear code's parity-check matrix $H$ (rows = classical parity checks, as in notebook 04) and turn *each row* into two quantum stabilizers — one built entirely from $X$s on that row's support, one entirely from $Z$s.

Row $h_i$ as an $X$-type stabilizer and row $h_j$ as a $Z$-type stabilizer commute exactly when they overlap in an even number of positions, i.e. when $h_i \cdot h_j = 0 \pmod 2$. So the whole construction works — *every* $X$-type generator commutes with *every* $Z$-type generator — precisely when

$$H H^T \equiv 0 \pmod 2.$$

This is a property of a *classical* matrix, not a quantum experiment, so it stays plain Python — there is nothing to simulate here.

In [7]:
def bits_of(j, width):
    return tuple((j >> shift) & 1 for shift in reversed(range(width)))


# Same construction as notebook 04: H's columns are the binary representations of 1..7.
H_columns = [bits_of(j, width=3) for j in range(1, 8)]
H = tuple(tuple(col[row] for col in H_columns) for row in range(3))

H_np = np.array(H)
HHt_mod2 = (H_np @ H_np.T) % 2
print("H =")
for row in H:
    print(row)
print("H H^T mod 2 =")
print(HHt_mod2)
print("All zero (checks are pairwise orthogonal):", np.all(HHt_mod2 == 0))

H =
(0, 0, 0, 1, 1, 1, 1)
(0, 1, 1, 0, 0, 1, 1)
(1, 0, 1, 0, 1, 0, 1)
H H^T mod 2 =
[[0 0 0]
 [0 0 0]
 [0 0 0]]
All zero (checks are pairwise orthogonal): True


The Hamming code's parity-check matrix satisfies $HH^T \equiv 0 \pmod 2$ — it is **dual-containing**. That single fact is the entire reason the *same* $H$ can generate both the $X$-type and $Z$-type stabilizers of a quantum code — no second classical code is needed.

## 6. The Steane code: CSS applied to the Hamming code, as an actual circuit

The Steane code's encoding circuit is the *quantum* version of notebook 04's classical `encode` function. Recall `encode` set the 4 message bits at positions 3, 5, 6, 7 (1-indexed) and computed 3 parity bits from them by XOR. The quantum circuit does exactly the same thing, with `H` in place of "free choice" and `CX` in place of XOR:

1. `H` on the 4 message-qubit positions (0-indexed 2, 4, 5, 6) — puts all $2^4=16$ possible messages into an equal superposition, instead of picking just one.
2. `CX` gates computing each parity qubit from its message qubits, following notebook 04's exact parity formulas ($p_1 = m_1\oplus m_2\oplus m_4$, and so on) — reversible XOR, so it works coherently on a superposition.

The result is $|0_L\rangle = \frac{1}{4}\sum_{c \,\in\, \text{Hamming code}} |c\rangle$ — an equal superposition of all 16 classical Hamming codewords from notebook 04.

In [8]:
def build_steane_encoder():
    sim = stim.TableauSimulator()
    # Message qubits: 0-indexed positions 2, 4, 5, 6 (1-indexed 3, 5, 6, 7).
    sim.h(2)
    sim.h(4)
    sim.h(5)
    sim.h(6)
    # p1 (position 0) = m1(2) xor m2(4) xor m4(6)
    sim.cx(2, 0); sim.cx(4, 0); sim.cx(6, 0)
    # p2 (position 1) = m1(2) xor m3(5) xor m4(6)
    sim.cx(2, 1); sim.cx(5, 1); sim.cx(6, 1)
    # p4 (position 3) = m2(4) xor m3(5) xor m4(6)
    sim.cx(4, 3); sim.cx(5, 3); sim.cx(6, 3)
    return sim


def row_to_pauli_string(row, letter, n=7):
    return "".join(letter if bit else "I" for bit in row)


steane_x_strings = [row_to_pauli_string(row, "X") for row in H]
steane_z_strings = [row_to_pauli_string(row, "Z") for row in H]

steane_sim = build_steane_encoder()
print("X-type stabilizer eigenvalues:")
for s in steane_x_strings:
    print(f"  {s}: {steane_sim.peek_observable_expectation(stim.PauliString(s)):+.0f}")
print("Z-type stabilizer eigenvalues:")
for s in steane_z_strings:
    print(f"  {s}: {steane_sim.peek_observable_expectation(stim.PauliString(s)):+.0f}")

X-type stabilizer eigenvalues:
  IIIXXXX: +1
  IXXIIXX: +1
  XIXIXIX: +1
Z-type stabilizer eigenvalues:
  IIIZZZZ: +1
  IZZIIZZ: +1
  ZIZIZIZ: +1


All 6 eigenvalues are $+1$: this circuit genuinely prepares the Steane code's $|0_L\rangle$, built entirely from notebook 04's $H$, with $k = 7 - 6 = 1$ logical qubit.

### Verification: direct commutation check

Confirm every $X$-type generator commutes with every $Z$-type generator using `stim.PauliString.commutes()` — a direct, literal check of notebook 03's stabilizer definition, and of Section 5's $HH^T=0$ claim.

In [9]:
all_commute = True
for sx in steane_x_strings:
    for sz in steane_z_strings:
        commute = stim.PauliString(sx).commutes(stim.PauliString(sz))
        all_commute &= commute
        print(f"[{sx}, {sz}] commute: {commute}")

print("\nEvery X-type generator commutes with every Z-type generator:", all_commute)

[IIIXXXX, IIIZZZZ] commute: True
[IIIXXXX, IZZIIZZ] commute: True
[IIIXXXX, ZIZIZIZ] commute: True
[IXXIIXX, IIIZZZZ] commute: True
[IXXIIXX, IZZIIZZ] commute: True
[IXXIIXX, ZIZIZIZ] commute: True
[XIXIXIX, IIIZZZZ] commute: True
[XIXIXIX, IZZIIZZ] commute: True
[XIXIXIX, ZIZIZIZ] commute: True

Every X-type generator commutes with every Z-type generator: True


### Prediction 2

Notebook 04 showed that a classical single-bit error's syndrome, under this exact $H$, directly names the error position as a binary number. **Before running the next cell:** for a single $X$ error on qubit 3 (0-indexed) of the Steane code, which stabilizers should fire — the $X$-type or the $Z$-type ones? What 3-bit syndrome (from those checks) do you expect, and how does it relate to notebook 04's syndrome table?

In [10]:
def eigen_to_bit(eigenvalue):
    return 0 if eigenvalue > 0 else 1


for qubit in range(7):
    sim = build_steane_encoder()
    sim.x(qubit)
    x_syndrome = tuple(eigen_to_bit(sim.peek_observable_expectation(stim.PauliString(s))) for s in steane_x_strings)
    z_syndrome = tuple(eigen_to_bit(sim.peek_observable_expectation(stim.PauliString(s))) for s in steane_z_strings)
    print(f"X error on qubit {qubit}: X-type syndrome (should be all 0) = {x_syndrome}, Z-type syndrome (locates it) = {z_syndrome}")

X error on qubit 0: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (0, 0, 1)
X error on qubit 1: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (0, 1, 0)
X error on qubit 2: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (0, 1, 1)
X error on qubit 3: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (1, 0, 0)
X error on qubit 4: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (1, 0, 1)
X error on qubit 5: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (1, 1, 0)
X error on qubit 6: X-type syndrome (should be all 0) = (0, 0, 0), Z-type syndrome (locates it) = (1, 1, 1)


An $X$ error is only visible to the $Z$-type checks (same-letter Paulis commute), and — because this circuit was built directly from notebook 04's $H$ — the resulting 3-bit syndrome is *exactly* notebook 04's classical syndrome for an error at that position, digit for digit. A $Z$ error would instead show up only in the $X$-type syndrome, by the mirror argument. The Steane code is, quite literally, two independent copies of the classical Hamming decoder — one watching for $X$ errors, one for $Z$ errors — bolted together by the CSS construction.

## Exercises

**1.** Write `demangle_degenerate_z_error(qubit_in_block_c)` that builds `build_shor_encoder()`, applies a $Z$ error to the given qubit (one of 6, 7, or 8 — block C), and applies a *fixed* correction (always `sim.z(6)`) regardless of which of the three actually happened. Confirm (via `state_vector()` and `same_state`) it restores the clean $|0_L\rangle$ state exactly in all three cases — this is Section 3's degeneracy argument, made into a working corrector.

**2.** Write `verify_css_condition(H)` that takes any binary parity-check matrix (as a tuple of tuples, like `H` in Section 5) and returns whether $HH^T \equiv 0 \pmod 2$. Try it on `H` from this notebook (should be `True`), and then on a parity-check matrix you construct for the $[3,1,3]$ repetition code from notebook 02 (a single row, `(1,1,0)` say) — does the repetition code's check matrix satisfy the CSS condition? What would that imply about building a "repetition-code CSS quantum code" the same way this notebook built the Steane code?

In [11]:
def demangle_degenerate_z_error(qubit_in_block_c):
    pass  # TODO: implement, then remove this stub body


def verify_css_condition(H):
    pass  # TODO: implement, then remove this stub body


# TODO: once implemented, try e.g.:
# clean = build_shor_encoder().state_vector()
# for qubit in (6, 7, 8):
#     corrected = demangle_degenerate_z_error(qubit)
#     print(qubit, same_state(corrected, clean))
# print(verify_css_condition(H))